# 10 · Model size: does Kronos' window-mean reversion depend on model size? (Colab T4)

Kronos-mini (4.1M) and Kronos-small (24.7M) vs the Kronos-base (102M) runs already on Drive.
Same 29 equities, lookbacks 64 and 256, plus the synthetic random-walk test. Exploratory (not pre-registered).
Every step resumes after a disconnect: rerun **Setup**, then the step. No secrets needed. Total ≈ 1–1.5 GPU-hours.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'
KRONOS = '/content/Kronos'

# Unpack code from Drive (auto-updates once for this notebook's new scripts).
import os, shutil, zipfile, sys
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/scripts/run_modelsize.py'):
    assert os.path.exists(ZIP), f'upload the new kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

!test -d $KRONOS || git clone -q https://github.com/shiyu-coder/Kronos $KRONOS
%pip install -q einops safetensors ccxt yfinance
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)

import glob
EQ = ' '.join(sorted(os.path.relpath(p, CODE) for p in glob.glob(f'{CODE}/configs/kronos_b/*.yaml')))
SMALL = 'model_id=NeoQuasar/Kronos-small'
MINI = 'model_id=NeoQuasar/Kronos-mini;tokenizer_id=NeoQuasar/Kronos-Tokenizer-2k'
!nvidia-smi -L
print(len(EQ.split()), 'equity configs')

## Step 1 · Synthetic random walks, small and mini (GPU, ~10 min)

In [ ]:
!cd $CODE && python scripts/run_synthetic.py --root $ROOT --kronos-path $KRONOS --model-id NeoQuasar/Kronos-small --lookback 64 256
!cd $CODE && python scripts/run_synthetic.py --root $ROOT --kronos-path $KRONOS --model-id NeoQuasar/Kronos-mini --tokenizer-id NeoQuasar/Kronos-Tokenizer-2k --lookback 64 256

## Step 2 · Kronos-small, lookbacks 64 and 256, 29 equities (GPU, ~35 min)

In [ ]:
!cd $CODE && python scripts/run_sampling.py --root $ROOT --kronos-path $KRONOS --config $EQ --set lookback=64 --set $SMALL
!cd $CODE && python scripts/run_sampling.py --root $ROOT --kronos-path $KRONOS --config $EQ --set lookback=256 --set $SMALL

## Step 3 · Kronos-mini, lookbacks 64 and 256, 29 equities (GPU, ~25 min)

In [ ]:
M = ' '.join(f'--set {s}' for s in MINI.split(';'))
!cd $CODE && python scripts/run_sampling.py --root $ROOT --kronos-path $KRONOS --config $EQ --set lookback=64 $M
!cd $CODE && python scripts/run_sampling.py --root $ROOT --kronos-path $KRONOS --config $EQ --set lookback=256 $M

## Step 4 · Reports (CPU, ~5 min) — paste both outputs back

In [ ]:
!cd $CODE && python scripts/run_phaseb.py --root $ROOT --config $EQ --out modelsize --run base64 "lookback=64" --run base256 "lookback=256" --run small64 "lookback=64;$SMALL" --run small256 "lookback=256;$SMALL" --run mini64 "lookback=64;$MINI" --run mini256 "lookback=256;$MINI"
!cd $CODE && python scripts/run_modelsize.py --root $ROOT